# Refactored Clustering, Stacking, and pPXF Pipeline
This notebook runs the refactored analysis pipeline using the reusable `src` package.

In [ ]:
from pathlib import Path
from src import ClusteringConfig, ModelParameters, PipelinePaths, RandomForestHyperparameters
from src.pipelines.full import FullPipeline
from src.data import load_inspire_master_catalogue

paths = PipelinePaths()
rf_params = RandomForestHyperparameters(
    n_estimators=50,
    max_depth=8,
    max_features=0.8,
    max_samples=0.7,
    min_samples_leaf=3,
    min_samples_split=5,
    random_state=1,
)

feature_display_names = {
    'MgFe': r'$[\mathrm{Mg}/\mathrm{Fe}]$ (dex)',
    '[M/H]_mean_mass': r'$[\mathrm{M}/\mathrm{H}]$ (dex)',
    '[M/H]_err_mass': r'$[\mathrm{M}/\mathrm{H}]$ error (dex)',
    'velDisp_ppxf_res': r'$\sigma_{\star}$ (km/s)',
    'velDisp_ppxf_err_res': r'$\sigma_{\star}$ error (km/s)',
    'age_mean_mass': r'Age (Gyr)',
    'age_err_mass': r'Age error (Gyr)',
}

model_params = ModelParameters(
    features=[
        'MgFe',
        '[M/H]_mean_mass',
        '[M/H]_err_mass',
        'velDisp_ppxf_res',
        'velDisp_ppxf_err_res',
        'age_mean_mass',
        'age_err_mass',
    ],
    small_boundary=0.45,
    large_boundary=0.6,
    hyperparameters=rf_params,
    feature_display_names=feature_display_names,
    combined_plot_filename='regression_performance_45_6_3regions.pdf',
    residuals_plot_filename='residuals_45_6_3regions.pdf',
    importance_plot_filename='feature_importance_45_6_3regions.pdf',
    predictions_csv=Path('outputs/cluster_results/regression_clusters.csv'),
)

config = ClusteringConfig(paths=paths, model=model_params)
pipeline = FullPipeline(config)


In [ ]:
inspire_df = load_inspire_master_catalogue(paths)


In [ ]:
result = pipeline.run(inspire_df, clean_outputs=True)
result


In [ ]:
import pandas as pd

rows = []
for method, summaries in result.stacking.items():
    for summary in summaries:
        rows.append({
            'method': method,
            'cluster_label': summary.cluster_label,
            'n_spectra': summary.n_spectra,
            'mean_DoR': summary.mean_dor,
            'sigma_fin': summary.sigma_fin,
        })
pd.DataFrame(rows)


In [ ]:
result.clustering.plot_paths
